# Velamuri 2018 — 2D Scalar FEM / Complex PML Bend Mode Research

**Paper:** [Study of Bending Losses in Optical Fibers using COMSOL](https://www.comsol.com/paper/study-of-bending-losses-in-optical-fibers-using-comsol-multiphysics-61111) (2018), Ashitosh Velamuri et al.

**Research scope:** Real 2D **scalar triangular P1 FEM** with complex stretched-coordinate PML. It solves for complex \(n_\mathrm{eff}\), instead of requiring the user to supply its imaginary part. **It is NOT the paper's full-vector COMSOL EWFD/GEBT simulation**. An optional beam/Poisson stress proxy does not replace actual GEBT.

The mesh is a shortened square pure-silica window plus PML; **physical 125 µm diameter cladding, coating, measured RIP and manufacturing stress are not modeled**. Results are not validated product bend-loss numbers. Strong sensitivity to mesh/PML/window is possible.

**Do not compare raw results to the paper's 0.0119 dB/turn** until the exact BIF index profile and all relevant mechanics/mesh conditions have been reproduced.

## 1. Pin the exact repository FEM source

This notebook embeds the current GitHub `solver.py` source directly for reproducibility. **No GitHub authentication or live download is required**, although NumPy, SciPy and Matplotlib need to be available. In Colab, run the cell below first.

In [ ]:
from pathlib import Path
Path("bend2d_solver.py").write_text("\"\"\"2D scalar P1 triangular FEM with complex PML for optical-fiber bend modes.\n\nResearch prototype inspired by Velamuri et al. (COMSOL Conference 2018).\nThis is a real 2-D eigenmode numerical solve, NOT their full-vector EWFD/GEBT,\nnor a verified model of any commercial cable or actual physical coating.\nThe PML is on a shortened glass-only computational window: do NOT interpret\nits leakage rate as certified macrobend loss without mesh/PML/window sweeps.\n\nEquation: div(C grad E) + k0^2 n_eq^2 s_x s_y E = beta^2 s_x s_y E\nC=diag(s_y/s_x, s_x/s_y), n_eq(x,y)=n_stress(r)(1+x/R).\nOuter boundary is zero after the complex-coordinate-stretched PML.\n\"\"\"\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict, replace\nfrom math import pi, log\nimport numpy as np\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.linalg import eigs, splu, LinearOperator\n\n_B = (.6961663,.4079426,.8974794)\n_C = (.0684043**2,.1162414**2,9.896161**2)\n\ndef silica_index(wavelength_nm):\n    w = wavelength_nm / 1000.\n    if not (.21 < w < 3.7):\n        raise ValueError(\"silica Sellmeier wavelength requires 210-3700 nm\")\n    return np.sqrt(1 + sum(b*w*w/(w*w-c) for b,c in zip(_B,_C)))\n\n@dataclass(frozen=True)\nclass Fiber:\n    wavelength_nm:float = 1550.\n    core_radius_um:float = 4.1\n    delta_n_core:float = .005\n    profile:str = \"step\"\n    trench_start_um:float = 7.\n    trench_end_um:float = 12.\n    trench_delta_n:float = -.002\n    dip_radius_um:float = 0.\n    dip_delta_n:float = 0.\n    bend_diameter_mm:float = 10.\n    stress_mode:str = \"none\"\n    poisson_ratio:float = .17\n    p11:float = .113\n    p12:float = .252\n    window_um:float = 22.\n    pml_thickness_um:float = 10.85\n    pml_strength:float = 2.\n    pml_exponent:int = 3\n    grid_h_um:float = .7\n    n_modes:int = 10\n    max_points:int = 220\n\ndef validate(f):\n    if f.profile not in (\"step\",\"trench\"):\n        raise ValueError(\"profile must be step or trench\")\n    if f.stress_mode not in (\"none\",\"beam_proxy\"):\n        raise ValueError(\"stress_mode must be none or beam_proxy (NOT GEBT)\")\n    for key in (\"wavelength_nm\",\"core_radius_um\",\"delta_n_core\",\"bend_diameter_mm\",\n                \"window_um\",\"pml_thickness_um\",\"pml_strength\",\"grid_h_um\"):\n        if not np.isfinite(getattr(f,key)):\n            raise ValueError(\"invalid \"+key)\n    if f.core_radius_um <= 0 or f.delta_n_core <= 0 or f.window_um <= 2*f.core_radius_um:\n        raise ValueError(\"need positive core/index step and window >2*core radius\")\n    if not 0 <= f.bend_diameter_mm <= 1000:\n        raise ValueError(\"bend diameter must be 0..1000 mm, zero is straight\")\n    if min(f.pml_thickness_um,f.pml_strength,f.grid_h_um) <= 0:\n        raise ValueError(\"PML thickness/strength and mesh size must be positive\")\n    if not 2 <= f.n_modes <= 15:\n        raise ValueError(\"n_modes must be 2..15\")\n    if f.stress_mode == \"beam_proxy\" and not 0 <= f.poisson_ratio < .5:\n        raise ValueError(\"invalid Poisson ratio\")\n    if f.profile == \"trench\" and not (\n        f.core_radius_um < f.trench_start_um < f.trench_end_um < f.window_um\n    ):\n        raise ValueError(\"trench must lie inside the non-PML computational window\")\n    if f.bend_diameter_mm and (\n        f.window_um + f.pml_thickness_um >= 500*f.bend_diameter_mm\n    ):\n        raise ValueError(\"2D computational half-width must be smaller than bend radius\")\n    silica_index(f.wavelength_nm)\n    return f\n\ndef index_field(f,x,y):\n    r=np.hypot(x,y)\n    n=silica_index(f.wavelength_nm) + f.delta_n_core*(r < f.core_radius_um)\n    if f.profile==\"trench\":\n        n=n+f.trench_delta_n*((r>=f.trench_start_um)&(r<f.trench_end_um))\n    if f.dip_radius_um>0:\n        n=n+f.dip_delta_n*(r<f.dip_radius_um)\n    if f.bend_diameter_mm>0:\n        eps=x/(f.bend_diameter_mm*500.)\n        if f.stress_mode==\"beam_proxy\":\n            n=n*(1-.5*n*n*(f.p11*eps-2*f.p12*f.poisson_ratio*eps))\n        n=n*(1+eps)\n    return n\n\ndef stretch(f,x):\n    frac=np.maximum((np.abs(x)-f.window_um)/f.pml_thickness_um,0)\n    return 1+1j*f.pml_strength*frac**f.pml_exponent\n\ndef assemble(f):\n    validate(f)\n    half=f.window_um+f.pml_thickness_um\n    nx=max(11,int(np.ceil(2*half/f.grid_h_um))+1)\n    if nx%2==0: nx+=1\n    if nx>f.max_points:\n        raise ValueError(\"mesh too large; increase grid_h_um or max_points\")\n    xx=np.linspace(-half,half,nx)\n    h=float(xx[1]-xx[0])\n    i,j=np.meshgrid(np.arange(nx-1),np.arange(nx-1),indexing=\"ij\")\n    v=(i*nx+j).ravel()\n    triangles=np.vstack([\n        np.stack([v,v+nx,v+nx+1],axis=1),\n        np.stack([v,v+nx+1,v+1],axis=1)])\n    nt=len(triangles)\n    X,Y=np.meshgrid(xx,xx,indexing=\"ij\")\n    coords=np.stack([X.ravel(),Y.ravel()],axis=1)\n    pts=coords[triangles]\n    x,y=pts[:,:,0],pts[:,:,1]\n    cross=((x[:,1]-x[:,0])*(y[:,2]-y[:,0])-\n           (x[:,2]-x[:,0])*(y[:,1]-y[:,0]))\n    area=np.abs(cross)/2\n    gx=np.stack([y[:,1]-y[:,2],y[:,2]-y[:,0],y[:,0]-y[:,1]],axis=1)/cross[:,None]\n    gy=np.stack([x[:,2]-x[:,1],x[:,0]-x[:,2],x[:,1]-x[:,0]],axis=1)/cross[:,None]\n    xc,yc=x.mean(axis=1),y.mean(axis=1)\n    sx,sy=stretch(f,xc),stretch(f,yc)\n    b=sx*sy\n    k0=2*pi/(f.wavelength_nm/1000)\n    n=index_field(f,xc,yc)\n    M=(area*b)[:,None,None]*(np.ones((3,3))+np.eye(3))[None,:,:]/12\n    A=(k0*k0*n*n)[:,None,None]*M-area[:,None,None]*(\n        (sy/sx)[:,None,None]*gx[:,:,None]*gx[:,None,:]+\n        (sx/sy)[:,None,None]*gy[:,:,None]*gy[:,None,:])\n    nodes=nx*nx\n    rows=np.broadcast_to(triangles[:,:,None],(nt,3,3)).ravel()\n    cols=np.broadcast_to(triangles[:,None,:],(nt,3,3)).ravel()\n    matA=coo_matrix((A.ravel(),(rows,cols)),shape=(nodes,nodes)).tocsr()\n    matM=coo_matrix((M.ravel(),(rows,cols)),shape=(nodes,nodes)).tocsr()\n    idx=np.arange(nx)[:,None]\n    jdx=np.arange(nx)[None,:]\n    interior=(idx>0)&(idx<nx-1)&(jdx>0)&(jdx<nx-1)\n    keep=np.flatnonzero(interior.ravel())\n    return matA[keep,:][:,keep],matM[keep,:][:,keep],coords[keep],nx,h\n\ndef solve(f, include_field=False):\n    A,M,coords,nx,h=assemble(f)\n    k0=2*pi/(f.wavelength_nm/1000)\n    n_guess=silica_index(f.wavelength_nm)+.45*f.delta_n_core\n    sigma=(k0*n_guess)**2\n    # For complex PML mass, SciPy eigs(A,M=M,sigma) is not safe: M is\n    # non-Hermitian/non-SPD. Apply generalized shift-invert explicitly.\n    lu=splu((A-sigma*M).tocsc())\n    operator=LinearOperator(A.shape,dtype=complex,matvec=lambda v:lu.solve(M@v))\n    mu,vecs=eigs(operator,k=f.n_modes,which=\"LM\",tol=1e-9,maxiter=1500)\n    beta_sq=sigma+1/mu\n    neffs=np.sqrt(beta_sq)/k0\n    r=np.hypot(coords[:,0],coords[:,1])\n    actual=(np.abs(coords).max(axis=1)<f.window_um)\n    nearcore=(r<2*f.core_radius_um)\n    modes=[]\n    for idx,neff in enumerate(neffs):\n        power=np.abs(vecs[:,idx])**2\n        core_fraction=float(power[nearcore].sum()/max(power[actual].sum(),1e-30))\n        physical_fraction=float(power[actual].sum()/max(power.sum(),1e-30))\n        modes.append(dict(index=idx,neff_real=float(neff.real),\n            neff_imag=float(neff.imag),core_fraction=core_fraction,\n            physical_fraction=physical_fraction))\n    selected=max(modes,key=lambda mode:mode[\"core_fraction\"])\n    imag=selected[\"neff_imag\"]\n    alpha_db_m=20/log(10)*k0*1e6*imag\n    bend_db_turn=(alpha_db_m*pi*f.bend_diameter_mm/1000\n                  if f.bend_diameter_mm>0 and imag>=0 else None)\n    result=dict(\n        input=asdict(f),neff_real=selected[\"neff_real\"],neff_imag=imag,\n        bend_loss_dB_per_turn=bend_db_turn,loss_dB_per_m=alpha_db_m,\n        core_fraction=selected[\"core_fraction\"],\n        physical_fraction=selected[\"physical_fraction\"],\n        grid_n=nx,h_um=h,unknowns=A.shape[0],ranked_modes=modes,\n        nonnegative_imag=bool(imag>=0),is_fully_validated=False,\n        solver_level=\"2D scalar P1 triangular FEM with complex PML (NOT full-vector COMSOL/GEBT)\",\n        limitations=[\n            \"Square PML truncates actual 125um cladding; coating not modeled\",\n            \"Requires independent PML thickness, mesh, domain and mode-tracking convergence\",\n            \"Optional linear beam/Poisson stress proxy is NOT GEBT\",\n            \"No experimental product or paper profile-level validation\"])\n    if include_field:\n        chosen=selected[\"index\"]\n        # Resize only the DISPLAY intensity; the mode was computed on the full mesh.\n        intensity=np.abs(vecs[:,chosen].reshape((nx-2,nx-2)))**2\n        intensity=intensity/max(float(intensity.max()),1e-30)\n        stride=max(1,int(np.ceil((nx-2)/101)))\n        result[\"mode_intensity_image\"]=intensity[::stride,::stride].tolist()\n        result[\"mode_image_stride\"]=stride\n        result[\"mode_image_origin_um\"]=-f.window_um-f.pml_thickness_um+h\n        result[\"mode_image_h_um\"]=h*stride\n    return result\n\ndef convergence_study(f,mesh_values=(.8,.65,.55),\n                      pml_values=(6.2,9.3,10.85),tolerance_percent=2.):\n    \"\"\"Real independent FEM reruns; not an assertion of convergence.\"\"\"\n    base=solve(f)\n    mesh=[dict(grid_h_requested_um=h,**solve(replace(f,grid_h_um=h)))\n          for h in mesh_values]\n    pml=[dict(pml_thickness_requested_um=t,\n              **solve(replace(f,pml_thickness_um=t))) for t in pml_values]\n    def compare(a,b):\n        x=a[\"bend_loss_dB_per_turn\"];y=b[\"bend_loss_dB_per_turn\"]\n        if x is None or y is None or y<=0:return None\n        return 100*abs(x-y)/abs(y)\n    mesh_change=compare(mesh[-2],mesh[-1]) if len(mesh)>=2 else None\n    pml_change=compare(pml[-2],pml[-1]) if len(pml)>=2 else None\n    passed=(mesh_change is not None and pml_change is not None\n        and mesh_change<=tolerance_percent and pml_change<=tolerance_percent)\n    return dict(base=base,mesh=mesh,pml=pml,\n                mesh_last_two_change_pct=mesh_change,\n                pml_last_two_change_pct=pml_change,\n                tolerance_percent=tolerance_percent,\n                last_two_sweeps_passed=passed,\n                is_fully_validated=False,\n                caveat=\"Even a pass requires domain-size, polarization, mode tracking, cladding and experimental convergence.\")\n\nif __name__==\"__main__\":\n    import argparse,json,time\n    parser=argparse.ArgumentParser(description=__doc__)\n    for name,field in Fiber.__dataclass_fields__.items():\n        val=field.default\n        parser.add_argument(\"--\"+name.replace(\"_\",\"-\"),type=type(val),default=val)\n    parser.add_argument(\"--convergence\",action=\"store_true\",\n                        help=\"repeat independent runs on 3 grids and 3 PML thicknesses\")\n    args=vars(parser.parse_args())\n    isstudy=args.pop(\"convergence\")\n    f=Fiber(**args)\n    start=__import__(\"time\").time()\n    result=convergence_study(f) if isstudy else solve(f)\n    result[\"elapsed_s\"]=__import__(\"time\").time()-start\n    print(json.dumps(result,indent=2))\n", encoding="utf-8")
from bend2d_solver import Fiber, solve, convergence_study, silica_index, index_field
from dataclasses import replace, asdict
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
print("2D triangular scalar P1 FEM / complex PML loaded; source version pinned in this notebook.")

## 2. Set a demonstrative model, not the paper's real material

Below is a **G.652-like illustrative Step-index surrogate**, with core radius 4.1 µm, \(\Delta n=0.005\), 1550 nm, bend **diameter 10 mm** (radius 5 mm). The paper's standard-fiber example has core radius 3.05 µm and a central dip, while its BIF uses a trench. Exact profiles were not supplied to this notebook.

Use `profile="trench"` with trench dimensions to test a *hypothetical* bend-insensitive profile, but do not label it real BIF validation.

In [ ]:
config=Fiber(wavelength_nm=1550, core_radius_um=4.1, delta_n_core=0.005,
    profile="step", bend_diameter_mm=10, stress_mode="none",
    window_um=22, pml_thickness_um=10.85, pml_strength=2.0,
    grid_h_um=0.7, n_modes=10)
straight=solve(replace(config,bend_diameter_mm=0),include_field=True)
bent=solve(config,include_field=True)
print("Straight neff:",straight['neff_real'],straight['neff_imag'])
print("Bent neff:",bent['neff_real'], "+",bent['neff_imag'],"j")
print("Bend loss (UNVERIFIED):",bent['bend_loss_dB_per_turn'],"dB/turn")
print("Mode core-power selection fraction:",bent["core_fraction"])
print("This is NOT a validated manufacturer's bending-loss result.")

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.3))
for ax,case,name in zip(axes,[straight,bent],["Straight","Bent — 10 mm diameter"]):
    field=np.array(case["mode_intensity_image"],dtype=float)
    dx=case["mode_image_h_um"];x0=case["mode_image_origin_um"]
    extent=[x0,x0+dx*(field.shape[0]-1),x0,x0+dx*(field.shape[1]-1)]
    ax.imshow(field.T,origin="lower",extent=extent,cmap="inferno",vmin=0,vmax=1,aspect="equal")
    ax.set(xlabel="Bend-direction x (µm)",ylabel="y (µm)",title=name)
    ax.axvline(0,linestyle="--",alpha=.3,color="cyan")
plt.suptitle("Calculated 2D scalar eigenmode intensity |E|² (not vector EWFD)")
plt.tight_layout();plt.show()

## 3. Independent mesh and PML convergence (mandatory)

Run several **new FEM solves**, not merely postprocessing one imported eigenvalue. The criterion below is user-selected **2% relative difference of the last two points**, not a standard. It does **not** check the domain window, coating, stress or the measured product.

**A failed criterion means the displayed loss is NOT numerically converged.** A passed criterion still does not certify the physical model.

In [ ]:
study=convergence_study(config,mesh_values=(0.8,0.65,0.55),
    pml_values=(6.2,9.3,10.85),tolerance_percent=2.0)
mesh_df=pd.DataFrame([{"requested_h_um":x["grid_h_requested_um"],"grid_n":x["grid_n"],
    "Im_neff":x["neff_imag"],"bend_loss_db_turn":x["bend_loss_dB_per_turn"]} for x in study["mesh"]])
pml_df=pd.DataFrame([{"pml_um":x["pml_thickness_requested_um"],
    "pml_lambda":x["pml_thickness_requested_um"]/(config.wavelength_nm/1000),
    "grid_n":x["grid_n"],"Im_neff":x["neff_imag"],
    "bend_loss_db_turn":x["bend_loss_dB_per_turn"]} for x in study["pml"]])
print("Mesh refinement:");display(mesh_df)
print("PML thickness sweep:");display(pml_df)
print("Mesh last-two change (%)",study["mesh_last_two_change_pct"])
print("PML last-two change (%)",study["pml_last_two_change_pct"])
print("Two checks meet 2% target?",study["last_two_sweeps_passed"])
if not study["last_two_sweeps_passed"]:
    print("NOT CONVERGED: do not report this value as a validated bending-loss prediction.")

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].plot(mesh_df.requested_h_um,mesh_df.bend_loss_db_turn,"o-")
axs[0].invert_xaxis()
axs[0].set(xlabel="Grid spacing h (µm) (smaller=refined)",ylabel="Bend loss (dB/turn)",title="Mesh sensitivity")
axs[1].plot(pml_df.pml_lambda,pml_df.bend_loss_db_turn,"o-")
axs[1].set(xlabel="PML thickness / wavelength",ylabel="Bend loss (dB/turn)",title="PML sensitivity")
for ax in axs:ax.grid(alpha=.3)
plt.tight_layout();plt.show()

## 4. Paper reference vs independent computation

The paper cites simulated BIF loss **0.0119 dB/turn** for **10 mm bend diameter at 1550 nm**, and mean experimental loss approximately **0.012 dB/turn**. That applies to the paper's specially designed BIF and its vector FEM/GEBT setup, **not** this hypothetical Step-index fiber.

Our model should *not* be fitted to 0.0119 by changing the core/trench arbitrarily and then called independent validation. Only supplied measured RIP, GEBT strain field, exterior optical index and converged full-vector PML can support a physical comparison.

In [ ]:
# Save a browser-importable record with explicit provenance and convergence warnings.
# Keep only summary metrics; plot intensity already shown above.
def compact(d):
    return {key:d[key] for key in ("input","neff_real","neff_imag",
      "bend_loss_dB_per_turn","loss_dB_per_m","core_fraction","physical_fraction",
      "grid_n","h_um","unknowns","nonnegative_imag",
      "is_fully_validated","solver_level","limitations")}
export={"file_kind":"LSCNS_BEND_2D_SCALAR_FEM_PML_V1","source":"Velamuri 2018 inspired research prototype",
  "base":compact(bent),"straight":compact(straight),
  "mesh":[compact(z) for z in study["mesh"]],
  "pml":[compact(z) for z in study["pml"]],
  "mesh_last_two_change_pct":study["mesh_last_two_change_pct"],
  "pml_last_two_change_pct":study["pml_last_two_change_pct"],
  "last_two_sweeps_passed":study["last_two_sweeps_passed"],
  "tolerance_percent":study["tolerance_percent"],
  "is_fully_validated":False,
  "warnings":["scalar P1 not full vector","no GEBT",
   "actual 125µm physical cladding/coating not modeled",
   "do not interpret as measured YOFC/LE125 data",
   "sweep criterion is not full domain or physical model validation"]}
out=Path("bend2d_pml_result.json");out.write_text(json.dumps(export,indent=2,ensure_ascii=False))
print(out.resolve())
try:
  from google.colab import files
  files.download(str(out))
except ImportError:
  print("Open the saved JSON and import it into the existing web workbench.")